In [7]:
import pandas as pd
import numpy as np
import joblib

pd.set_option("display.max_columns", None)

# Load the model created by occupancy_prediction.ipynb
model_bundle = joblib.load("occupancy_model.joblib")

model = model_bundle["model"]
features = model_bundle["features"]
USER_LAT = model_bundle["user_lat"]
USER_LON = model_bundle["user_lon"]

print("Occupancy model loaded successfully.")
print("Features:", features)
print("User location:", USER_LAT, USER_LON)


Occupancy model loaded successfully.
Features: ['price', 'avg_rating', 'distance_km']
User location: 25.6242129 85.0850287


In [8]:
# Load the datasets needed for recommendation
occupancy = pd.read_csv("dataset/lot_occupancy_hourly.csv")
parking = pd.read_csv("dataset/parking_lots.csv")
reviews = pd.read_csv("dataset/reviews.csv")

# Create average rating for each parking lot
rating_df = (
    reviews.groupby("lot_id", as_index=False)
    .agg(
        avg_rating=("rating", "mean"),
        review_count=("rating", "count")
    )
)

# Build the parking-lot recommendation dataset
df = (
    occupancy
    .merge(
        parking[
            [
                "parking_lots_id", "name", "address", "latitude", "longitude",
                "price_per_hour", "total_slots", "vehicle_type", "status"
            ]
        ],
        left_on="lot_id",
        right_on="parking_lots_id",
        how="left"
    )
    .merge(rating_df, on="lot_id", how="left")
)

df["avg_rating"] = df["avg_rating"].fillna(reviews["rating"].mean())
df["price"] = df["price_per_hour"]

print("Merged shape:", df.shape)


Merged shape: (4575, 23)


In [9]:
# Calculate distance from the same user location stored with the occupancy model
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(
        np.radians, [lat1, lon1, lat2, lon2]
    )
    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    )
    return 2 * R * np.arcsin(np.sqrt(a))

df["distance_km"] = haversine_km(
    USER_LAT,
    USER_LON,
    df["latitude"].values,
    df["longitude"].values
)


In [10]:
# One row per parking lot
lot_features = (
    df[
        [
            "lot_id", "name", "address",
            "price", "avg_rating", "distance_km"
        ]
    ]
    .drop_duplicates("lot_id")
    .copy()
)

# IMPORTANT: this prediction comes from occupancy_model.joblib
lot_features["predicted_occupancy_pct"] = np.clip(
    model.predict(lot_features[features]),
    0,
    100
)

lot_features["predicted_available_pct"] = (
    100 - lot_features["predicted_occupancy_pct"]
)

# Keep only approved parking lots
recommendations = lot_features.merge(
    parking[["parking_lots_id", "status", "vehicle_type", "total_slots"]],
    left_on="lot_id",
    right_on="parking_lots_id",
    how="left"
)

recommendations = recommendations[
    recommendations["status"] == "approved"
].copy()

recommendations.head()


,lot_id,name,address,price,avg_rating,distance_km,predicted_occupancy_pct,predicted_available_pct,parking_lots_id,status,vehicle_type,total_slots
0,P115,Pitampura Parking E,"Pitampura, Delhi",28.57,3.863636,857.449203,1.376127,98.623873,P115,approved,all,25
1,P132,Saket Parking B,"Saket, Delhi",16.68,4.000000,844.160574,1.426085,98.573915,P132,approved,ev,78
2,P108,Lajpat Nagar Parking C,"Lajpat Nagar, Delhi",40.16,3.909091,842.069338,2.847814,97.152186,P108,approved,all,51
3,P122,Saket Parking B,"Saket, Delhi",18.80,3.888889,843.241549,2.300000,97.700000,P122,approved,bike,21
4,P110,Pitampura Parking E,"Pitampura, Delhi",17.62,3.872727,858.048657,2.317855,97.682145,P110,approved,ev,17


In [11]:
#Recommendation score
'''
 **40%** availability
- **25%** rating
- **20%** distance
- **15%** price

Higher score = better recommendation.
'''



def minmax(series):
    mn, mx = series.min(), series.max()
    if mx == mn:
        return pd.Series(1.0, index=series.index)
    return (series - mn) / (mx - mn)

recommendations["availability_score"] = minmax(
    100 - recommendations["predicted_occupancy_pct"]
)

recommendations["rating_score"] = (
    recommendations["avg_rating"] / 5
)

recommendations["distance_score"] = (
    1 - minmax(recommendations["distance_km"])
)

recommendations["price_score"] = (
    1 - minmax(recommendations["price"])
)

recommendations["recommendation_score"] = (
    0.40 * recommendations["availability_score"]
    + 0.25 * recommendations["rating_score"]
    + 0.20 * recommendations["distance_score"]
    + 0.15 * recommendations["price_score"]
) * 100

top_parking = (
    recommendations
    .sort_values("recommendation_score", ascending=False)
    [[
        "name", "address", "price", "avg_rating",
        "distance_km", "predicted_occupancy_pct",
        "predicted_available_pct", "recommendation_score",
        "vehicle_type", "total_slots"
    ]]
    .head(5)
)

top_parking


,name,address,price,avg_rating,distance_km,predicted_occupancy_pct,predicted_available_pct,recommendation_score,vehicle_type,total_slots
1,Saket Parking B,"Saket, Delhi",16.68,4.000000,844.160574,1.426085,98.573915,91.594055,ev,78
8,Nehru Place Parking D,"Nehru Place, Delhi",24.00,3.883721,840.396934,1.561049,98.438951,89.578790,ev,56
3,Saket Parking B,"Saket, Delhi",18.80,3.888889,843.241549,2.300000,97.700000,83.647312,bike,21
9,Saket Parking B,"Saket, Delhi",34.67,3.921053,843.993891,2.034599,97.965401,78.611356,all,51
25,Greater Kailash Parking B,"Greater Kailash, Delhi",40.44,3.841270,841.915066,2.122604,97.877396,76.568915,bike,57


In [12]:
# Best parking recommendation
best = top_parking.iloc[0]

print("Recommended Parking:")
print("Name:", best['name'])
print("Address:", best["address"])
print(f"Price: ₹{best['price']:.2f}/hour")
print(f"Rating: {best['avg_rating']:.2f}/5")
print(f"Distance: {best['distance_km']:.2f} km")
print(f"Predicted occupancy: {best['predicted_occupancy_pct']:.2f}%")
print(f"Predicted available: {best['predicted_available_pct']:.2f}%")
print(f"Recommendation score: {best['recommendation_score']:.2f}/100")


Recommended Parking:
Name: Saket Parking B
Address: Saket, Delhi
Price: ₹16.68/hour
Rating: 4.00/5
Distance: 844.16 km
Predicted occupancy: 1.43%
Predicted available: 98.57%
Recommendation score: 91.59/100
